# Desk-quarter ops exceptions: grouped count SearchSession

Synthetic **desk × quarter** panel (teaching twin of a sealed CoS challenge).
Each row is one desk in one quarter. The response `exception_count` is a
non-negative integer with a long right tail. Desks repeat over time, so
rows that share a `desk_id` are dependent — folds and the holdout must
respect that unit.

This case walks the full native SearchSession loop on a **count / panel /
MAR** problem:

1. Exhaustive `pre_scan` (`native` / `neural` / `dual`)
2. Domain reasoning (ops drivers vs collinear proxies)
3. Grouped folds, NBinomial mean model, measured `compare` / `adopt`
4. Full interaction path: `interaction_screen` → `pair_candidates` → `tree_candidate`
5. `tune` → `freeze` → `final_evaluate`

The mboostrs screens propose terms. `compare` measures their error changes
on shared grouped folds, and `adopt` applies the two-standard-error rule.

The sealed DGP answer key is **not** reproduced here; sealed grades live
under the private challenge tree / `.remember` logs for CoS runs.


:::{admonition} What this study estimates
:class: dropdown

| | |
|---|---|
| **Target** | predict next-window settlement/ops exception counts for held-out desks |
| **Observed response** | `exception_count` (non-negative integer) |
| **Estimand** | conditional mean count given desk-quarter covariates (predictive) |
| **Units** | exception events per desk-quarter |
| **Sampling unit** | one desk in one quarter |
| **Dependence** | repeated quarters within `desk_id`; group holdout and grouped CV |
| **Split unit** | entire desks held out (~20%); training uses 4 grouped folds on `desk_id` |
| **Tuning** | `tune` after measured adopts; complete-case rows (see API friction note) |
| **Uncertainty** | paired fold SE on specification `change`; adopt iff \(|\Delta| > 2\,\mathrm{SE}\) |
| **Claim type** | predictive / methodological — count panel SearchSession with MAR and proxies |

:::


## Library map

| Role | mboostrs call | Notes |
|------|---------------|-------|
| Tree probe | `pre_scan(engine='native')` | permutation / PD / H-statistic pairs |
| Neural probe | `pre_scan(engine='neural')` | native neural nomination |
| Dual probe | `pre_scan(engine='dual')` | agreement; keep report for `pair_candidates(surrogate=...)` |
| Residual pairs | `interaction_screen` then `pair_candidates` | surrogate must be report dict or `False` (#31) |
| Multiway tree | `tree_candidate` | nomination, not adoption |
| Holdout | `freeze` → `final_evaluate` | needs a **response-scale** score (#40) |

**API friction (this run):** `na_action='omit'` currently mismatches residual
length vs the full frame in `interaction_screen` / `pair_candidates` / `tune`
([#39](https://github.com/fcela/mboostrs/issues/39)). The reproducible driver
uses complete-case rows and `score='mse'` so the full interaction path and
holdout complete. Prefer fixing omit alignment over dropping rows in
production analyses.


In [1]:
from pathlib import Path
import json
import polars as pl

ROOT = Path("..").resolve() if Path("bench_desk_ops_exceptions.json").exists() else Path("docs/book").resolve()
# notebook may run from docs/book/cases or repo root
for cand in [Path("bench_desk_ops_exceptions.json"), Path("docs/book/cases/bench_desk_ops_exceptions.json")]:
    if cand.exists():
        BENCH = cand
        break
else:
    BENCH = Path("docs/book/cases/bench_desk_ops_exceptions.json")

bench = json.loads(BENCH.read_text())
print("train_rows", bench["dataset"]["train_rows"], "test_rows", bench["dataset"]["test_rows"])
print("family", bench["search"].get("family"), "mstop", bench["search"].get("final_mstop"))
print("final_spec:", bench["search"].get("final_spec"))
print("holdout:", bench.get("holdout_metrics"))
print("gaps_observed:", len(bench.get("gaps_observed") or []))


train_rows 2647 test_rows 650
family NBinomial mstop 183
final_spec: bols(quarter) + bols(region) + bols(aum_bn) + bbs(complexity_score, df=4) + bols(overnight_frac) + bols(vendor_n) + bols(has_legacy_stack) + bols(staff_per_bn) + bols(vol_proxy) + bols(rate_env) + bols(util_ratio) + bbs(vendor_n, by=has_legacy_stack)
holdout: {'rmse': 6.590886150298107, 'mae': 2.3414108966989886, 'mean_poisson_deviance': 15.354325392940616, 'n': 650}
gaps_observed: 2


## Domain reasoning (recorded before adopt)

Ops reading used in the driver transcript:

- **Drivers:** complexity, overnight settlement share, legacy stack, vendor depth, staffing per AUM, book size, region.
- **Red herrings:** `aum_bn_proxy` and `complexity_z` are near-collinear siblings — keep one of each pair in the baseline (#34).
- **Family:** count with variance ≫ mean → start from overdispersed NBinomial rather than Poisson.
- **Interactions:** expect non-product structure (thresholds / hinges). Nominate via the full residual + tree path; adopt only on two-SE improvement.

The full experimental rerun uses a development-only driver. From the
`mboostrs-dev` checkout:

```bash
RAYON_NUM_THREADS=1 .venv/bin/python tools/run_desk_ops_exceptions.py
```


In [2]:
# Compact live sketch (short mstop) — full transcript is in the bench JSON.
import mboostrs as mb
from mboostrs.search_session import open_search
from mboostrs.policy import CostBudget

data_path = None
for cand in [Path("../data/desk_ops_exceptions.parquet"), Path("docs/book/data/desk_ops_exceptions.parquet")]:
    if cand.exists():
        data_path = cand
        break
panel = pl.read_parquet(data_path)
train = panel.filter(pl.col("split") == "train").drop_nulls()
test = panel.filter(pl.col("split") == "test").drop_nulls()

session = open_search(
    train, "exception_count",
    family=mb.NBinomial(), score="mse",
    groups="desk_id", folds=4, seed=20260929,
    mstop=40, nu=0.3, budget=CostBudget(fits=60),
)
session.run("set_spec", rhs=(
    "bols(region) + bols(aum_bn) + bbs(complexity_score, df=4) "
    "+ bols(overnight_frac) + bols(vendor_n) + bols(has_legacy_stack) "
    "+ bols(staff_per_bn)"
))
session.run("fit")
ix = session.run("interaction_screen")
pairs = session.run("pair_candidates", surrogate=False)  # not True — see #31
tree = session.run("tree_candidate")
print("top interaction_screen pairs:", (ix.summary or {}).get("top_pairs", [])[:3])
print("pair leads:", (pairs.summary or {}).get("leads", [])[:2])
print("tree term:", (tree.summary or {}).get("term"))
session.run("freeze")
final = session.run("final_evaluate", data=test)
print("holdout mse:", final.summary)


top interaction_screen pairs: [{'left': 'aum_bn', 'right': 'complexity_z', 'removed': 2.146076258249984, 'thresholded_sibling_suggested': False, 'threshold_column': None, 'sibling_term': None, 'next_action': None}, {'left': 'aum_bn_proxy', 'right': 'complexity_z', 'removed': 2.0104850280744646, 'thresholded_sibling_suggested': False, 'threshold_column': None, 'sibling_term': None, 'next_action': None}, {'left': 'aum_bn', 'right': 'complexity_score', 'removed': 1.813148805503868, 'thresholded_sibling_suggested': False, 'threshold_column': None, 'sibling_term': None, 'next_action': None}]
pair leads: [{'term': 'bbs(aum_bn, df=2.5, knots=5) %X% bbs(complexity_score, df=2.5, knots=5)', 'source': 'residual', 'kind': 'tensor'}, {'term': 'bbs(aum_bn, df=2.5, knots=5) %X% bbs(vendor_n, df=2.5, knots=5)', 'source': 'residual', 'kind': 'tensor'}]
tree term: btree(region, aum_bn, complexity_score, overnight_frac, vendor_n, has_legacy_stack, staff_per_bn, maxdepth=5, selection=sse)
holdout mse: {'

/Users/fcela/src/public_github/mboostrs/python/mboostrs/_autospec.py:2945: UserWarning: Pearson residuals are not defined for NBinomial; falling back to the response-scale residual y - predict(type='response').
  residual = pearson_residual(fit, data, response)


## Adopted specification and holdout scores

The reproduced bench run keeps an additive NBinomial ops baseline, then
adopts the varying-coefficient term `bbs(vendor_n, by=has_legacy_stack)`
(candidate `pair_4`) when it clears the two-SE rule among nominated
tensors / trees; the final fit uses 183 boosting steps.
Proxies are compared as red-herring checks and not auto-adopted.

Holdout on complete-case test rows (from the bench JSON) is reported as
RMSE / MAE / mean Poisson deviance alongside `final_evaluate`'s MSE.

For CoS sealed grading against the hidden DGP, see the private challenge
`GRADE_vs_DGP.md` and `.remember/logs/gap-hunt-20260929.md` — not pasted
here so the panel remains reusable as a teaching challenge.
